## nb_03_2_game_silver

Cleans and validates `bronze.game` and writes the result to `silver.game`.
Every cleaning/validation rule lives in its own function (defined once,
below) and is then applied one step at a time in its own cell, so each
intermediate result can be inspected before moving to the next step.

### Imports

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, count, when, lit, min, max,
    substring, concat, to_timestamp, to_date,
)

StatementMeta(, 4df791b4-26f0-46d5-ba04-87f285561d22, 3, Finished, Available, Finished, False)

### Load common db functions

In [2]:
%run nb_00_dbutils

StatementMeta(, 4df791b4-26f0-46d5-ba04-87f285561d22, 16, Finished, Available, Finished, True)

### Parameters

`RUN_PIPELINE` controls whether the "Run the pipeline" steps below actually execute.
It defaults to `True` for normal, standalone runs of this notebook.

When this notebook is loaded from another notebook via `%run` (e.g. from a test
notebook), pass `RUN_PIPELINE = False` as a run parameter so only the function/config
definitions are loaded and the pipeline against the real `bronze.game` / `silver.game`
tables is skipped:

```
%run <notebook name> { "RUN_PIPELINE": false }
```

In [3]:
# This cell is tagged "parameters" so Fabric/Synapse can override it when the
# notebook is invoked with %run nb_02_game_silver { "RUN_PIPELINE": false }
RUN_PIPELINE: bool = True

StatementMeta(, 4df791b4-26f0-46d5-ba04-87f285561d22, 17, Finished, Available, Finished, False)

### Config

In [4]:
BRONZE_TABLE = "bronze.game"
SILVER_TABLE = "silver.game"

# Only these columns make it into the silver table
required_cols: list[str] = [
    "game_id",
    "season",
    "type",
    "date_time_GMT",
    "home_team_id",
    "away_team_id",
    "home_goals",
    "away_goals",
]

# Natural key used to de-duplicate rows
DEDUPE_KEYS: list[str] = ["game_id"]
PRIMARY_KEYS: list[str] = ["game_id"]

# Valid game "type" values: A = All-Star, R = Regular season, P = Playoffs
VALID_TYPES: tuple[str] = ("A", "R", "P")


# this prevents table from loading when called from test script
if RUN_PIPELINE:
    TEAM_INFO_HOME_FOREIGN_KEYS = ["home_team_id", "team_id"]
    TEAM_INFO_AWAY_FOREIGN_KEYS = ["away_team_id", "team_id"]
    team_info = load_table(spark, "silver.team_info", columns=["team_id"])


StatementMeta(, 4df791b4-26f0-46d5-ba04-87f285561d22, 18, Finished, Available, Finished, False)

✅ Loaded silver.team_info: 33 rows


### Function: validate date_time_GMT to season
1. Confirm it falls within its own season's date range
   (a season runs roughly Sept 15 of the start year through Sept 30 of the end year).
2. Convert date_time_GMT to a date-only column.
Rows failing either check are dropped and reported.

In [5]:
def validate_game_date_to_season(df: DataFrame) -> DataFrame:
    """Validate date_time_GMT (parseable + within season) and cast it to a date."""

    invalid_ts_count = df.filter(col("date_time_GMT").isNull()).count()
    if invalid_ts_count > 0:
        print(f"⚠️ {invalid_ts_count} row(s) have an unparseable date_time_GMT — dropping them")
    parsed = df.filter(col("date_time_GMT").isNotNull())

    # Must fall within the season's date range
    with_bounds = (
        parsed
        .withColumn("season_start", substring(col("season"), 1, 4).cast("int"))
        .withColumn("season_end", substring(col("season"), 5, 4).cast("int"))
        .withColumn(
            "season_start_date",
            to_timestamp(concat(col("season_start"), lit("-09-15 00:00:00"))),
        )
        .withColumn(
            "season_end_date",
            to_timestamp(concat(col("season_end"), lit("-09-30 23:59:59"))),
        )
    )

    invalid_season = with_bounds.filter(
        (col("date_time_GMT") < col("season_start_date"))
        | (col("date_time_GMT") > col("season_end_date"))
    )
    invalid_season_count = invalid_season.count()
    if invalid_season_count > 0:
        print(f"⚠️ {invalid_season_count} row(s) have date_time_GMT outside their season — dropping them")
        invalid_season.select("game_id", "season", "date_time_GMT").show(truncate=False)

    valid = with_bounds.filter(
        (col("date_time_GMT") >= col("season_start_date"))
        & (col("date_time_GMT") <= col("season_end_date"))
    )

    # Convert to date-only and drop the helper columns
    valid = (
        valid
        .withColumn("date_time_GMT", to_date(col("date_time_GMT")))
        .drop("season_start", "season_end", "season_start_date", "season_end_date")
    )
    return valid

StatementMeta(, 4df791b4-26f0-46d5-ba04-87f285561d22, 19, Finished, Available, Finished, False)

## Run the pipeline
Each step runs in its own cell so the result can be inspected before moving on.

In [6]:
if RUN_PIPELINE:
    df = load_table(spark, BRONZE_TABLE, columns=required_cols)

    df = remove_duplicates(df, columns=DEDUPE_KEYS)
    df = validate_no_nulls(df, columns=required_cols)
    df = validate_column_values(df, column="type", values=VALID_TYPES)

    # Convert date_time_GMT to date
    df = df.withColumn(
        "date_time_GMT",
        to_timestamp(col("date_time_GMT"))
    )

    # Validate game date against seasons
    df = validate_game_date_to_season(df)

    # Validate and drop incorrect foreign keys
    df = drop_foreign_key_violations(df, ftable=team_info, keys=TEAM_INFO_HOME_FOREIGN_KEYS)
    df = drop_foreign_key_violations(df, ftable=team_info, keys=TEAM_INFO_AWAY_FOREIGN_KEYS)

    df = validate_foreign_keys(df, ftable=team_info, keys=TEAM_INFO_HOME_FOREIGN_KEYS)
    df = validate_foreign_keys(df, ftable=team_info, keys=TEAM_INFO_AWAY_FOREIGN_KEYS)    

    # Validate primary key
    df = validate_primary_keys(df, keys=PRIMARY_KEYS)

    write_table(df, SILVER_TABLE)
    print("🏁 Silver load complete.")

StatementMeta(, 4df791b4-26f0-46d5-ba04-87f285561d22, 20, Finished, Available, Finished, False)

✅ Loaded bronze.game: 26305 rows
🔁 Removed 2570 duplicate row(s) based on ['game_id']
✅ Primary key check passed — ['game_id'] is unique across 23735 row(s)
⚠️ Dropping 5 rows with foreign key violations for ['home_team_id', 'team_id']
✅ Foreign key check passed — ['away_team_id', 'team_id']
✅ Foreign key check passed — ['home_team_id', 'team_id']
✅ Foreign key check passed — ['away_team_id', 'team_id']
✅ Wrote silver.game (23730 rows, 8 columns)
🏁 Silver load complete.
